# Empty, null, and absent values

How `None` / `null`, a missing key, `""`, `[]`, and `{}` are scored by default,
and how to change that with the three escape hatches: **transform**, **custom
comparator**, and **post-handler**.

See `06_example_postprocess.ipynb` for a
deeper look at the post-handler on its own.

Reminder of how a status counts toward metrics:

| status | precision | recall |
|--------|-----------|--------|
| match / mismatch | yes | yes |
| omission | no | yes |
| hallucination | yes | no |
| skipped | excluded | excluded |

Key idea: in the default eval an **omission is a missing key** (no key -> no
value). A key that *is present* but holds `null` is therefore **not** an
omission -- the value `null` is compared, which at a leaf is a **mismatch** if the gold is a real value, and a **match** if the gold is also `null`.

## Part 1 - Default scoring

One record, where each field isolates one situation from the empty/absent
family.

In [ ]:
from struct_extract_eval import evaluate
from struct_extract_eval.core.comparators.comparator import ComparatorResult
from struct_extract_eval.core.comparators.registry import register as register_comparator
from struct_extract_eval.core.transforms.registry import register as register_transform
from struct_extract_eval.postprocess import NullHandling, reclassify_nulls


def show(title, result):
    """Print every field result for the first record as a table."""
    print(title)
    print(f"  P={result.mean_precision:.2f}  R={result.mean_recall:.2f}  F1={result.mean_f1:.2f}")
    print(f"    {'field':<12}{'gold':<18}{'extracted':<14}{'status':<14}score")
    for fr in result.records[0].field_results:
        print(f"    {fr.path:<12}{fr.gold_value!r:<18}{fr.extracted_value!r:<14}{fr.status:<14}{fr.score}")


SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string", "x-eval-compare": "exact"},    # value vs null
        "note": {"type": "string", "x-eval-compare": "exact"},     # null vs null
        "code": {"type": "string", "x-eval-compare": "exact"},     # value vs ""
        "temp": {"type": "number", "x-eval-compare": "numeric"},   # value vs null (numeric)
        "tags": {"type": "array", "items": {"type": "string", "x-eval-compare": "exact"}},      # values vs []
        "aliases": {"type": "array", "items": {"type": "string", "x-eval-compare": "exact"}},   # [] vs []
        "meta": {"type": "object", "properties": {"author": {"type": "string", "x-eval-compare": "exact"}}},  # object vs null
        "extra": {"type": "object", "properties": {"k": {"type": "string", "x-eval-compare": "exact"}}},      # {} vs {}
    },
}

GOLD = [{
    "title": "Anneal", "note": None, "code": "X1", "temp": 500,
    "tags": ["a", "b"], "aliases": [], "meta": {"author": "me"}, "extra": {},
}]
EXTRACTED = [{
    "title": None,   # key present, value null
    "note": None,    # null on both sides
    "code": "",      # empty string vs a real value
    "temp": None,    # null at a numeric leaf
    "tags": [],      # empty array vs two gold elements
    "aliases": [],   # empty array on both sides
    "meta": None,    # null where gold has an object
    "extra": {},     # empty object on both sides
}]

show("default", evaluate(GOLD, EXTRACTED, SCHEMA))

What each situation produces by default:

| field | gold | extracted | situation | status | explanation                              |
|-------|------|-----------|-----------|--------|------------------------------------------|
| `title` | "Anneal" | `None` | value vs null | **mismatch** | key present, value None                  |
| `note` | `None` | `None` | null vs null | **match** (`exact`) | expect None, get None, match             |
| `code` | "X1" | `""` | value vs empty string | **mismatch** | key present, value not match             |
| `temp` | 500 | `None` | value vs null (numeric) | **mismatch**
| `tags` | ["a","b"] | `[]` | values vs empty array | **2 omissions** |
| `aliases` | `[]` | `[]` | empty vs empty array | **match** |
| `meta` | {author} | `None` | object vs null | **omission** (`meta.author`) | key `author` inside container is missing |
| `extra` | `{}` | `{}` | empty vs empty object | **match** | both empty, agree (like `[]` vs `[]`)    |

Notes: `meta`=null is treated as absent at the container, so it expands to a
per-child omission; `{}` vs `{}` yields an object-level match
(like `[]` vs `[]`); `""` is a present value, so it mismatches a real gold.

## Part 2 - change the default behavior

### 2.1 - Transform override (`x-eval-transform`)

Transforms now receive `None` (the dispatcher no longer skips it), so a custom
transform can rewrite it. Here `none_to_empty` coalesces `None -> ""` so a null
and an empty string are treated as the same value. Scope: a single field's
*present* values.

In [ ]:
def none_to_empty(value, params):
    return "" if value is None else value


register_transform("none_to_empty", none_to_empty)

base = {"type": "object", "properties": {"code": {"type": "string", "x-eval-compare": "exact"}}}
with_transform = {
    "type": "object",
    "properties": {
        "code": {"type": "string", "x-eval-compare": "exact", "x-eval-transform": ["none_to_empty"]},
    },
}
GOLD_T = [{"code": ""}]
EXTRACTED_T = [{"code": None}]

show('without transform ("" vs None)', evaluate(GOLD_T, EXTRACTED_T, base))
show("with none_to_empty transform", evaluate(GOLD_T, EXTRACTED_T, with_transform))

`"" vs None` is a mismatch by default; after `none_to_empty` both sides
become `""` and it is a match.

### 2.2 - Custom comparator override (`x-eval-compare`)

A comparator receives the raw values, **including `None`**, so it defines a
per-field null policy. Here `nullable` treats `None` and `""` as equivalent
"absent" markers.

In [ ]:
def compare_nullable(gold, extracted, params):
    g = None if gold in (None, "") else gold
    e = None if extracted in (None, "") else extracted
    if g == e:
        return ComparatorResult(score=1.0, comparator="nullable")
    return ComparatorResult(score=0.0, comparator="nullable", reason=f"{gold!r} != {extracted!r}")


register_comparator("nullable", compare_nullable)

base = {"type": "object", "properties": {"code": {"type": "string", "x-eval-compare": "exact"}}}
with_nullable = {"type": "object", "properties": {"code": {"type": "string", "x-eval-compare": "nullable"}}}
GOLD_C = [{"code": None}]
EXTRACTED_C = [{"code": ""}]

show('default exact (None vs "")', evaluate(GOLD_C, EXTRACTED_C, base))
show("with nullable comparator", evaluate(GOLD_C, EXTRACTED_C, with_nullable))

## - Post-handler override (`evaluate(post_process=...)`)

A post-handler runs after scoring and rewrites statuses dataset-wide.
`reclassify_nulls` treats configured values as absent: present-but-null
mismatches become omissions, both-absent becomes skipped. See
`06_example_postprocess.ipynb` for the full treatment.

In [ ]:
config = NullHandling(absent_values=[None, ""], both_absent_skip=True)

show("default (no post-handler)", evaluate(GOLD, EXTRACTED, SCHEMA))
show(
    "with reclassify_nulls(absent=[None, ''])",
    evaluate(GOLD, EXTRACTED, SCHEMA, post_process=lambda frs: reclassify_nulls(frs, config)),
)

You can also write your own post-handler to override the null classification. for example, make `None` vs `None` to be a match.

## Which tool for which scope

| Want to... | Use |
|------------|-----|
| Update one field's *present* values (incl. rewrite `None`) before reaching comparator | **transform** (`x-eval-transform`) |
| Define a per-field rule comparing the two raw values (e.g. `None == ""`) | **custom comparator** (`x-eval-compare`) |
| Apply one null/empty policy across the whole dataset, incl. reaching missing keys | **post-handler** (`evaluate(post_process=...)`) |